# Long-form Nepali YouTube transcription (Colab)

This notebook downloads audio from a YouTube URL, transcribes it with a permissive Nepali Whisper model, applies conservative rule-based cleanup, and exports text, SRT, VTT, JSON, DOCX, and a ZIP bundle.

Use **Runtime > Change runtime type > T4 GPU**. The pipeline is resumable, so a disconnect does not discard completed chunks.

Typical limits on free Colab: roughly 8–12 hours of audio per session. For longer recordings, run it in parts or use a paid runtime. Keep outputs in Google Drive.

In [ ]:
#@title 1. Install dependencies and clone the repository
import os
import subprocess

REPO_URL = "https://github.com/DarpanAdhikari/youtube-caption.git"
REPO_DIR = "/content/youtube-caption"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("Repository:", os.getcwd())

subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)
subprocess.run(
    ["python", "-m", "pip", "install", "-q", "-r", "requirements-pipeline.txt"],
    check=True,
)

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
print(gpu.stdout.strip() or "No GPU detected; transcription will be slow on CPU.")
print(subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0])

In [ ]:
#@title 2. Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")

## Configure the job

The output folder and model cache are stored in Drive so they survive runtime resets. `MODEL_RUNTIME_CACHE` is the fast local copy; the pipeline copies the model to Drive after the first download.

The default model is a provisional choice. Run the benchmark cell before committing to a model for a large archive.

In [ ]:
#@title 3. Paths and imports
from pathlib import Path

from transcript_pipeline.config import PipelineConfig
from transcript_pipeline.pipeline import run_pipeline

OUTPUT_ROOT = Path("/content/drive/MyDrive/DRP-Transcripts")
MODEL_RUNTIME_CACHE = Path("/content/models")
MODEL_DRIVE_CACHE = OUTPUT_ROOT / "_models"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_RUNTIME_CACHE.mkdir(parents=True, exist_ok=True)
MODEL_DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
print("Output root:", OUTPUT_ROOT)

In [ ]:
#@title 4. Video URL
VIDEO_URL = "https://www.youtube.com/watch?v=REPLACE_WITH_VIDEO_ID"  #@param {type:"string"}
print(VIDEO_URL)

In [ ]:
#@title 5. Transcribe
config = PipelineConfig(
    video_url=VIDEO_URL,
    output_root=OUTPUT_ROOT,
    model="nepali-medium-ct2",
    fallback_model="openai-turbo",
    refine_mode="rule",
    run_mode="auto",
    device="auto",
    compute_type="float16",
    model_cache_dir=MODEL_RUNTIME_CACHE,
    drive_model_cache_dir=MODEL_DRIVE_CACHE,
)

result = run_pipeline(config)
print("Status:", result.job.status)
print("Output:", result.paths.root)
for name, path in result.exports.files.items():
    print(f"  {name}: {path}")
if result.job.warnings:
    print("Warnings:")
    for warning in result.job.warnings:
        print("  -", warning)

In [ ]:
#@title 6. Download the ZIP bundle
from google.colab import files

files.download(str(result.exports.files["bundle_zip"]))

## Resume after a disconnect

If the runtime stops, reconnect, run the setup/mount/config/URL cells again, then run the cell below. Completed chunks are not transcribed again. Use `run_mode="retry_failed"` to reset chunks that failed permanently.

To change the model or chunk settings, use `run_mode="fresh"` and `force_overwrite=True`; that intentionally discards the previous run state.

In [ ]:
#@title 7. Resume / retry / re-export
RUN_MODE = "resume"  #@param ["resume", "retry_failed", "export_only", "refine_only"]
config.run_mode = RUN_MODE
result = run_pipeline(config)
print(result.job.status, result.paths.root)

## Benchmark the candidate models

Upload a small set of representative recordings to `BENCH_AUDIO`. Upload matching reference `.txt` files to `BENCH_REFS` when available; files are matched by stem, for example `sample1.wav` and `sample1.txt`. You can also benchmark a URL directly with the CLI command in `transcript_pipeline/README.md`.

WER/CER are useful but not sufficient: inspect names, numbers, dates, constituencies, ministries, and procedural terms manually.

In [ ]:
#@title 8. Benchmark uploaded audio
from transcript_pipeline.benchmark import benchmark_models

BENCH_AUDIO = Path("/content/benchmark/audio")
BENCH_REFS = Path("/content/benchmark/references")
BENCH_OUT = OUTPUT_ROOT / "_benchmark" / "uploaded"
BENCH_MODELS = [
    "nepali-medium-ct2",
    "nepali-turbo-ct2",
    "nepali-large-v3",
    "openai-large-v3",
]

audio_paths = sorted(
    path for path in BENCH_AUDIO.glob("*")
    if path.suffix.lower() in {".wav", ".mp3", ".m4a", ".mp4", ".opus", ".webm"}
)
references = {
    path.stem: path.read_text(encoding="utf-8")
    for path in BENCH_REFS.glob("*.txt")
}
print(f"{len(audio_paths)} audio file(s), {len(references)} reference(s)")

if audio_paths:
    summary = benchmark_models(
        BENCH_MODELS,
        audio_paths,
        references,
        config,
        BENCH_OUT,
    )
    print("Summary:", BENCH_OUT / "summary.md")
else:
    print("Add audio files to", BENCH_AUDIO, "first.")


### Reference-backed FLEURS check

The cell below downloads 10 public FLEURS `ne_np` test clips with human-written references and runs WER/CER on the GPU. This is a sanity check for the pipeline and model, not a substitute for the parliament quality gate. For the parliament gate, upload 2–3 representative clips and matching transcripts to `BENCH_AUDIO` / `BENCH_REFS`, then rerun cell 8.

Run one large model at a time if Colab disk or RAM is tight; edit the model lists in the benchmark cells.

In [ ]:
#@title 9. Download FLEURS ne_np clips with references
import csv
import tarfile
from huggingface_hub import hf_hub_download

FLEURS_LIMIT = 10
FLEURS_DIR = Path("/content/benchmark/fleurs-ne")
FLEURS_DIR.mkdir(parents=True, exist_ok=True)
tsv_path = Path(
    hf_hub_download("google/fleurs", "data/ne_np/test.tsv", repo_type="dataset")
)
archive_path = Path(
    hf_hub_download(
        "google/fleurs",
        "data/ne_np/audio/test.tar.gz",
        repo_type="dataset",
    )
)

rows = []
seen = set()
with tsv_path.open(encoding="utf-8") as handle:
    for row in csv.reader(handle, delimiter="\t"):
        if len(row) < 7:
            continue
        text = row[3].strip()
        if not text or text in seen:
            continue
        seen.add(text)
        rows.append((row[1], text))
        if len(rows) >= FLEURS_LIMIT:
            break

with tarfile.open(archive_path, "r:gz") as archive:
    members = {
        Path(member.name).name: member
        for member in archive.getmembers()
        if member.name.endswith(".wav")
    }
    for filename, text in rows:
        member = members[filename]
        target = FLEURS_DIR / filename
        with archive.extractfile(member) as source, target.open("wb") as destination:
            destination.write(source.read())
        (FLEURS_DIR / f"{Path(filename).stem}.txt").write_text(
            text + "\n", encoding="utf-8"
        )

print(f"Prepared {len(rows)} FLEURS clips in {FLEURS_DIR}")


In [ ]:
#@title 10. Run reference-backed FLEURS benchmark
from transcript_pipeline.benchmark import benchmark_models

FLEURS_AUDIO = sorted(FLEURS_DIR.glob("*.wav"))
FLEURS_REFS = {
    path.stem: (FLEURS_DIR / f"{path.stem}.txt").read_text(encoding="utf-8")
    for path in FLEURS_AUDIO
}
FLEURS_OUT = OUTPUT_ROOT / "_benchmark" / "fleurs-ne"
FLEURS_MODELS = [
    "nepali-medium-ct2",
    "nepali-turbo-ct2",
    "nepali-large-v3",
    "openai-large-v3",
]

summary = benchmark_models(
    FLEURS_MODELS,
    FLEURS_AUDIO,
    FLEURS_REFS,
    config,
    FLEURS_OUT,
)

for model in FLEURS_MODELS:
    rows = [row for row in summary["rows"] if row["model"] == model]
    scored = [row for row in rows if row["wer"] is not None]
    if scored:
        mean_wer = sum(row["wer"] for row in scored) / len(scored)
        mean_cer = sum(row["cer"] for row in scored) / len(scored)
        print(f"{model:24} mean WER {mean_wer:.2%}  mean CER {mean_cer:.2%}")

print("Summary:", FLEURS_OUT / "summary.md")


## Model choices and practical notes

- `nepali-medium-ct2` is fast and small; it is the provisional default.
- `nepali-turbo-ct2` is larger and needs its provenance/quality verified on your audio.
- `nepali-large-v3` and `nepali-large-v3-kiran` are Transformers checkpoints and are converted to CT2 on first use; conversion is slow and needs disk space.
- `openai-turbo` and `openai-large-v3` are baselines, not Nepali specialists.
- The curated registry excludes non-commercial models.

For failed or weak chunks, set `keep_chunks=True` and `keep_wav=True` while debugging. They are deleted after a clean run otherwise.

In [ ]:
#@title 11. Show the model registry
from transcript_pipeline.models import list_models

for spec in list_models():
    print(f"{spec.key:24} {spec.license:12} {spec.repo_id}")

## Troubleshooting

- **CUDA out of memory:** use `nepali-medium-ct2`, set `compute_type="int8_float16"` or `"int8"`, and close other notebooks.
- **No GPU:** set `device="cpu"` and expect a long run; `compute_type` is forced to `int8` on CPU.
- **YouTube blocks the download:** try `--cookies` / `cookies_path`, a different network, or a local audio upload via `--audio`.
- **Job completed with errors:** inspect `state/errors.jsonl`; rerun with `run_mode="retry_failed"`.
- **Bad names/numbers:** add exact mappings to `corrections.json` and terms to `glossary.json`, then use `run_mode="refine_only"`.
- **Drive fills up:** remove old job folders and cached models; the runtime cache is temporary, the Drive cache is not.